# Singing mice vocal modes - PAG TeLC experiment

## Import Libraries

In [ ]:
# import libraries
import glob
import numpy as np
import os
import pandas as pd
from pathlib import Path
import pickle
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import seaborn as sns

In [ ]:
# import internal libraries

## Helper Functions

In [ ]:
# import internal libraries
from utils import (
    compute_1d_kde,
    compute_2d_kde,
    find_threshold_between_peaks,
    draw_ellipse,
)

from events import (
    compute_event_features,
    compute_window_features,
    merge_stereotypy_windows,
    characterize_segments,
    classify_segments,
    classify_notes_by_temporal_patterning,
    classify_notes_by_amplitude,
    find_vocal_bouts,
    analyze_song_patterning,
    get_song_social_context,
    assign_bout_duration_to_notes,
    characterize_song_context,
)

## Parameters

In [ ]:
path_to_nas = '/mnt/nas_s2d1'

# identify loud vs soft notes
amp_high_threshold = -5 # based on solosocial-social, separating the two modes
amp_song_threshold = -55

# identify local windows of stereotyped sequences
min_window_size = 7
window_max_log_isi_threshold = -1 # based on solosocial-social, separating local stereotypy vs not
window_rmse_threshold = 0.15 # based on solosocial-social, separating local stereotypy vs not

# merge local windows into long segments
segment_max_time_gap = 0.5
segment_max_index_gap = 5

# load the sklearn models for segment features
with open('segment_features_scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
# Load the PCA
with open('segment_features_pca.pkl', 'rb') as f:
    pca = pickle.load(f)
# Load the GMM
with open('segment_features_gmm.pkl', 'rb') as f:
    gmm = pickle.load(f)

cluster_mapping = {0: 'S2', 1: 'S3', 2: 'S1'}

# color palettes for plotting
color_palette_patterning = {
    'S1': 'mediumturquoise',
    'S2': 'royalblue',
    'S3': 'blueviolet',
    'U' : 'orchid',
}

# final clusters and mapping
final_type_mapping = {'S1': 'song', 'S2': 'proto-song', 'S3': 'USV', 'U': 'USV'}
final_color_palette_patterning = {
    'song': 'mediumturquoise',
    'proto-song': 'royalblue',
    'USV': 'purple',
}

song_final_note_occupancy = 0.5

In [ ]:
# plotting
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 6

## Load session info

In [ ]:
sessions_info_filepath = os.path.join(path_to_nas, 'usv_calls', '251007_dryad_upload', 'pag_telc', 'pag_telc_sessions.csv')
sessions_info = pd.read_csv(sessions_info_filepath, header=0)
sessions_info = sessions_info[(sessions_info['condition']=='pre')|(sessions_info['condition']=='post')].reset_index(drop=True)
sessions_info

In [ ]:
sessions_curated_hours_info_filepath = os.path.join(path_to_nas, 'usv_calls', '251007_dryad_upload', 'pag_telc', 'pag_telc_curation.csv')
sessions_curated_hours_info = pd.read_csv(sessions_curated_hours_info_filepath, header=0)
sessions_curated_hours_info

In [ ]:
# make color palette
sorted_animal_ids = sorted(sessions_info['mouse_1'].unique())
palette = sns.color_palette(n_colors=len(sorted_animal_ids))
palette_dict = dict(zip(sorted_animal_ids, palette))

In [ ]:
all_sessions = []
for _, session_info in sessions_info.iterrows():

    session = {}

    session_id = session_info['session']
    project_dir = session_info['basepath']
    condition = session_info['condition']
    mouse_1 = session_info['mouse_1']
    mouse_2 = session_info['mouse_2']
    anchor_file_idx = int(session_info['anchor_file_idx'])
    num_vocal_days = int(session_info['num_vocal_days'])
    num_post_days = session_info['num_post_days']
    session_basepath = os.path.join(path_to_nas, 'usv_calls', '251007_dryad_upload', 'pag_telc', session_id)
    print(f'Processing session: {session_basepath}')

    if condition == 'pre':
        audio_dir = 'process_audio'
    else:
        audio_dir = 'original_audio'
    base_dir = Path(f'{session_basepath}/{audio_dir}/all_notes')
    file_indices = sorted([p.name for p in base_dir.iterdir() if p.is_dir()])
    print(f'Num hours: {len(file_indices)}')

    # find curated hours
    session_curated_hours = sessions_curated_hours_info[sessions_curated_hours_info['session']==session_id]
    print(f'Number of curated hours in this session: {len(session_curated_hours)}')
    curated_hours_file_indices = session_curated_hours['hr'].values

    # fill metadata info
    session['session_id'] = session_id
    session['project_dir'] = project_dir
    session['session_basepath'] = session_basepath
    session['condition'] = condition
    session['anchor_file_idx'] = anchor_file_idx
    session['num_post_days'] = num_post_days
    session['num_vocal_days'] = num_vocal_days
    session['mouse_1'] = mouse_1
    session['mouse_2'] = mouse_2
    session['num_files'] = len(file_indices)
    session['num_curated_files'] = len(curated_hours_file_indices)

    # all hours

    # for each hour, get number of calls
    hours_info = []
    hours_all_notes = []
    hours_l_notes = []
    hours_r_notes = []
    for i_file, file_idx in enumerate(file_indices):
        
        # # get start time in world time
        # l_audio_filepath = glob.glob(os.path.join(session_basepath, 'original_audio', 'left', f'*_{file_idx}.wav'))[0]
        # audio_filename = os.path.splitext(os.path.basename(l_audio_filepath))[0]
        # audio_filename_timestamp = pd.to_datetime(audio_filename.split('_')[0][1:], format='%y%m%d%H%M%S')
        
        # get all notes
        if session['condition'] == 'pre':
            all_notes_dir = os.path.join(session_basepath, 'process_audio', 'all_notes', file_idx)
        else:
            all_notes_dir = os.path.join(session_basepath, 'original_audio', 'all_notes', file_idx)
            
        if os.path.exists((os.path.join(all_notes_dir, 'all_notes_corrected_features.csv'))):
            all_notes_filepath = os.path.join(all_notes_dir, 'all_notes_corrected_features.csv')
        elif os.path.exists((os.path.join(all_notes_dir, 'all_notes_corrected.csv'))):
            all_notes_filepath = os.path.join(all_notes_dir, 'all_notes_corrected.csv')
        elif os.path.exists((os.path.join(all_notes_dir, 'all_notes.csv'))):
            all_notes_filepath = os.path.join(all_notes_dir, 'all_notes.csv')
        else:
            print(f'Cannot find any all_notes feature file, skipping session: {session_id}, file_idx: {file_idx}')
            continue
        # print('reading file', all_notes_filepath)

        # load notes
        all_notes = pd.read_csv(all_notes_filepath, header=0, index_col=0)
        all_notes = all_notes.drop_duplicates(subset=['start', 'end'], keep='last')
        all_notes = all_notes.sort_values(by=['start', 'end']).reset_index(drop=True)

        all_notes['session_id'] = session['session_id']
        all_notes['file_idx'] = file_idx
        l_notes = all_notes[all_notes['code']==0].copy().reset_index(drop=True)
        r_notes = all_notes[all_notes['code']==1].copy().reset_index(drop=True)
        l_notes['mouse_id'] = session['mouse_1']
        l_notes['mouse_side'] = 'mouse_1'
        r_notes['mouse_id'] = session['mouse_2']
        r_notes['mouse_side'] = 'mouse_2'

        # save the hour's notes
        hours_all_notes.append(all_notes)
        hours_l_notes.append(l_notes)
        hours_r_notes.append(r_notes)

        # save the hour's info
        hours_info.append({
            'i_file': i_file,
            'file_idx': file_idx,
            # 'file_timestamp': audio_filename_timestamp,
            'all_notes_filepath': all_notes_filepath,
            'l_num_notes': len(l_notes),
            'r_num_notes': len(r_notes),
        })

    df_hours_info = pd.DataFrame(hours_info)
    session['hours_info'] = df_hours_info
    # session['session_start_timestamp'] = df_hours_info['file_timestamp'].min()
    # session['session_reference_day'] = session['session_start_timestamp'].date()
    session['hours_all_notes'] = hours_all_notes
    session['hours_l_notes'] = hours_l_notes
    session['hours_r_notes'] = hours_r_notes

    # curated hours
    curated_hours_info = []
    curated_hours_all_notes = []
    curated_hours_l_notes = []
    curated_hours_r_notes = []
    for _, file_idx in enumerate(curated_hours_file_indices):

        # # get start time in world time
        # l_audio_filepath = glob.glob(os.path.join(session_basepath, 'original_audio', 'left', f'*_{file_idx:07d}.wav'))[0]
        # audio_filename = os.path.splitext(os.path.basename(l_audio_filepath))[0]
        # audio_filename_timestamp = pd.to_datetime(audio_filename.split('_')[0][1:], format='%y%m%d%H%M%S')

        # get all notes
        if session['condition'] == 'pre':
            all_notes_dir = os.path.join(session_basepath, 'process_audio', 'all_notes', f'{file_idx:07d}')
        else:
            all_notes_dir = os.path.join(session_basepath, 'original_audio', 'all_notes', f'{file_idx:07d}')
            
        if os.path.exists((os.path.join(all_notes_dir, 'all_notes_corrected_features.csv'))):
            all_notes_filepath = os.path.join(all_notes_dir, 'all_notes_corrected_features.csv')
        else:
            print(f'Cannot find any curated all_notes feature file, skipping session: {session_id}, file_idx: {file_idx:07d}')
            continue
        # print('reading file', all_notes_filepath)

        # load notes
        all_notes = pd.read_csv(all_notes_filepath, header=0, index_col=0)
        all_notes = all_notes.drop_duplicates(subset=['start', 'end'], keep='last')
        all_notes = all_notes.sort_values(by=['start', 'end']).reset_index(drop=True)

        all_notes['session_id'] = session['session_id']
        all_notes['file_idx'] = file_idx
        l_notes = all_notes[all_notes['code']==0].copy().reset_index(drop=True)
        r_notes = all_notes[all_notes['code']==1].copy().reset_index(drop=True)
        l_notes['mouse_id'] = session['mouse_1']
        l_notes['mouse_side'] = 'mouse_1'
        r_notes['mouse_id'] = session['mouse_2']
        r_notes['mouse_side'] = 'mouse_2'

        # save the hour's notes
        curated_hours_all_notes.append(all_notes)
        curated_hours_l_notes.append(l_notes)
        curated_hours_r_notes.append(r_notes)

        # save the hour's info
        curated_hours_info.append({
            'i_file': np.where(np.array(file_indices).astype(int)==file_idx)[0][0],
            'file_idx': file_idx,
            # 'file_timestamp': audio_filename_timestamp,
            'all_notes_filepath': all_notes_filepath,
            'l_num_notes': len(l_notes),
            'r_num_notes': len(r_notes),
        })

    df_curated_hours_info = pd.DataFrame(curated_hours_info)
    session['curated_hours_info'] = df_curated_hours_info
    session['curated_hours_all_notes'] = curated_hours_all_notes
    session['curated_hours_l_notes'] = curated_hours_l_notes
    session['curated_hours_r_notes'] = curated_hours_r_notes

    # save the output
    all_sessions.append(session)

## Overview pre vs post

In [ ]:
# select overview hours
for session in all_sessions:

    overview_hours_info = []
    overview_hours_all_notes = []
    overview_hours_l_notes = []
    overview_hours_r_notes = []

    session_id = session['session_id']
    mouse_1 = session['mouse_1']
    mouse_2 = session['mouse_2']
    condition = session['condition']

    print(f'Processing session: {session_id}; mouse_1: {mouse_1}; condition: {condition}')
    # Process hours_info
    hours_info = session['hours_info'].copy()
    # hours_info['file_timestamp'] = pd.to_datetime(hours_info['file_timestamp'])
    # reference_day = pd.Timestamp(session['session_reference_day'])
    # hours_info['hour_offset'] = (hours_info['file_timestamp'] - reference_day).dt.total_seconds() / 3600.0
    
    # Get the anchor file index and look up the corresponding row
    anchor_file_idx = session["anchor_file_idx"]
    # # Get the anchor file index and look up the corresponding row
    # anchor_file_idx = session["anchor_file_idx"]
    # anchor_row = hours_info[hours_info['file_idx'].astype(int) == anchor_file_idx]
    # if anchor_row.empty:
    #     print(f"Anchor file idx {anchor_file_idx} not found for session {session['session_id']}")
    #     continue
    # anchor_hour_offset = anchor_row.iloc[0]['hour_offset']
    # print(f'anchor_file_idx: {anchor_file_idx}; time offset: {anchor_hour_offset:.2f}')
    
    anchor_row = hours_info[hours_info['file_idx'].astype(int) == anchor_file_idx]
    # anchor_row_index = anchor_row.index[0]
    # if anchor_row.empty:
    #     print(f"Anchor file idx {anchor_file_idx} not found for session {session['session_id']}")
    #     continue
    # print(f'anchor i_file: {anchor_row_index}')

    # For pre sessions, record day 1 only using the anchor hour
    if condition == 'pre':
        if anchor_row.empty:
            print(f"Anchor file idx {anchor_file_idx} not found for session {session['session_id']}")
            continue
        else:
            anchor_row_index = anchor_row.index[0]
            print(f'pre; file: {hours_info.iloc[anchor_row_index]["all_notes_filepath"]}')
            overview_hours_info.append(hours_info.iloc[anchor_row_index])
            overview_hours_all_notes.append(session['hours_all_notes'][anchor_row_index])
            overview_hours_l_notes.append(session['hours_l_notes'][anchor_row_index])
            overview_hours_r_notes.append(session['hours_r_notes'][anchor_row_index])
    
    # For post sessions, record data for 5 days at 24-hour increments starting at the anchor hour.
    else:
        for day in range(1, int(session['num_post_days']) + 1):
            target_file_index = anchor_file_idx + 24 * (day - 1)
            target_row = hours_info[hours_info['file_idx'].astype(int) == target_file_index]
            if target_row.empty:
                print(f"Target file idx {target_file_index} not found for session {session['session_id']} on day {day}")
                continue
            else:
                row_index = target_row.index[0]
                print(f'post day: {day}; file: {hours_info.iloc[row_index]["all_notes_filepath"]}')
                # target_time = anchor_hour_offset + 24 * day
                # row_index = (hours_info['hour_offset'] - target_time).abs().idxmin()
                # print(f'post day: {day}; hour offset: {hours_info.iloc[row_index]["hour_offset"] - anchor_hour_offset:.2f}; file: {hours_info.iloc[row_index]["all_notes_filepath"]}')
                overview_hours_info.append(hours_info.iloc[row_index])
                overview_hours_all_notes.append(session['hours_all_notes'][row_index])
                overview_hours_l_notes.append(session['hours_l_notes'][row_index])
                overview_hours_r_notes.append(session['hours_r_notes'][row_index])
    
    df_overview_hours_info = pd.DataFrame(overview_hours_info).reset_index(drop=True)
    session['overview_hours_info'] = df_overview_hours_info
    session['overview_hours_all_notes'] = overview_hours_all_notes
    session['overview_hours_l_notes'] = overview_hours_l_notes
    session['overview_hours_r_notes'] = overview_hours_r_notes

In [ ]:
# analyze temporal patterning
for session in all_sessions:
    print(f'Analyzing session: {session["session_id"]}')
    print(f'Num overview hours: {len(session["overview_hours_l_notes"])}')

    # Process left notes
    processed_overview_left_notes = []
    processed_overview_left_windows = []
    processed_overview_left_filtered_windows = []
    processed_overview_left_segments = []
    processed_overview_left_songs = []

    for hour_df in session['overview_hours_l_notes']:
        print(f'Num left notes: {len(hour_df)}')
        # compute temporal features - note level
        df_events = compute_event_features(hour_df)
        # compute temporal features - fixed size window
        df_windows = compute_window_features(df_events, min_window_size=min_window_size)
        # join fixed size windows into long segments
        df_filtered_windows, df_segments = merge_stereotypy_windows(df_windows,
                                                                    df_events,
                                                                    max_log_interval_threshold=window_max_log_isi_threshold,
                                                                    rmse_threshold=window_rmse_threshold,
                                                                    max_gap=segment_max_time_gap,
                                                                    max_index_gap=segment_max_index_gap)
        # characterize segments
        df_segments = characterize_segments(df_events, 
                                            df_segments, 
                                            ransac_min_samples=min_window_size,
                                            ransac_normalized_residual_threshold=window_rmse_threshold,)
        # classify segments
        df_segments = classify_segments(df_segments, scaler, pca, gmm, cluster_mapping=cluster_mapping)
        # classify notes by temporal patterning based on the segment it's in
        df_events = classify_notes_by_temporal_patterning(df_segments, df_events)
        # # classify notes by loudness
        # df_events = classify_notes_by_amplitude(df_events, amp_high_threshold)
        # consolidate vocalizations into final types
        df_events['type_final'] = df_events['type_patterning'].map(final_type_mapping)

        # analyze songs defined by the linear backbone of the song segment
        if df_segments.empty:
            df_songs = pd.DataFrame()
        else:
            df_song_segments = df_segments[df_segments['cluster_label']=='S1']
            # df_song_segments = df_segments[(df_segments['cluster_label']=='S1') | (df_segments['cluster_label']=='S2')]
            df_songs = analyze_song_patterning(df_song_segments, df_events, song_final_note_occupancy=song_final_note_occupancy)
        
        processed_overview_left_notes.append(df_events)
        processed_overview_left_windows.append(df_windows)
        processed_overview_left_filtered_windows.append(df_filtered_windows)
        processed_overview_left_segments.append(df_segments)
        processed_overview_left_songs.append(df_songs)

    session['overview_hours_l_notes'] = processed_overview_left_notes
    session['overview_hours_l_windows'] = processed_overview_left_windows
    session['overview_hours_l_filtered_windows'] = processed_overview_left_filtered_windows
    session['overview_hours_l_segments'] = processed_overview_left_segments
    session['overview_hours_l_songs'] = processed_overview_left_songs

In [ ]:
overview_hours = []

# Loop through sessions to collect data points
for session in all_sessions:

    session_id = session['session_id']
    mouse_1 = session['mouse_1']
    mouse_2 = session['mouse_2']
    condition = session['condition']

    print(f'Processing session: {session_id}; mouse_1: {mouse_1}; condition: {condition}')

    # Process hours_info
    hours_info = session['hours_info'].copy()
    # hours_info['file_timestamp'] = pd.to_datetime(hours_info['file_timestamp'])
    # reference_day = pd.Timestamp(session['session_reference_day'])
    # hours_info['hour_offset'] = (hours_info['file_timestamp'] - reference_day).dt.total_seconds() / 3600.0

    # Get the anchor file index and look up the corresponding row
    anchor_file_idx = session["anchor_file_idx"]
    # # Get the anchor file index and look up the corresponding row
    # anchor_file_idx = session["anchor_file_idx"]
    anchor_row = hours_info[hours_info['file_idx'].astype(int) == anchor_file_idx]
    # if anchor_row.empty:
    #     print(f"Anchor file idx {anchor_file_idx} not found for session {session['session_id']}")
    #     continue
    # anchor_hour_offset = anchor_row.iloc[0]['hour_offset']
    # print(f'anchor_file_idx: {anchor_file_idx}; time offset: {anchor_hour_offset:.2f}')

    df_overview_hours_info = session['overview_hours_info'].copy()

    # For pre sessions, record day 1 only using the anchor hour
    if condition == 'pre':
        row_index = df_overview_hours_info.iloc[0]['i_file']
        print(f'pre; row index: {row_index}; file: {hours_info.iloc[row_index]["all_notes_filepath"]}')
        l_num_notes = len(session['overview_hours_l_notes'][0])
        r_num_notes = len(session['overview_hours_r_notes'][0])
        l_notes = session['overview_hours_l_notes'][0]
        l_songs = session['overview_hours_l_songs'][0]
        l_num_usvs = len(l_notes[l_notes['type_final']=='USV'])
        l_num_songs = len(l_songs)
        # hour_offset = hours_info.iloc[row_index]['hour_offset']
        overview_hours.append({
            'session_id': session_id,
            'condition': condition,
            'mouse_1': mouse_1,
            'mouse_2': mouse_2,
            'day': 0,
            # 'hour_offset': hour_offset,
            'all_notes_filepath': hours_info.iloc[row_index]['all_notes_filepath'],
            'l_num_notes': l_num_notes,
            'r_num_notes': r_num_notes,
            'l_num_usvs': l_num_usvs,
            'l_num_songs': l_num_songs,
        })

    # For post sessions, record data for 5 days at 24-hour increments starting at the anchor hour.
    else:
        for i_day, overview_hour in df_overview_hours_info.iterrows():
            row_index = overview_hour['i_file']
            print(f'post day: {i_day+1}; file: {hours_info.iloc[row_index]["all_notes_filepath"]}')
            # hour_offset = hours_info.iloc[row_index]['hour_offset']
            l_num_notes = len(session['overview_hours_l_notes'][i_day])
            r_num_notes = len(session['overview_hours_r_notes'][i_day])
            l_notes = session['overview_hours_l_notes'][i_day]
            l_songs = session['overview_hours_l_songs'][i_day]
            l_num_usvs = len(l_notes[l_notes['type_final']=='USV'])
            l_num_songs = len(l_songs)
            overview_hours.append({
                'session_id': session_id,
                'condition': condition,
                'mouse_1': mouse_1,
                'mouse_2': mouse_2,
                'day': i_day+1,
                # 'hour_offset': hour_offset,
                'all_notes_filepath': hours_info.iloc[row_index]['all_notes_filepath'],
                'l_num_notes': l_num_notes,
                'r_num_notes': r_num_notes,
                'l_num_usvs': l_num_usvs,
                'l_num_songs': l_num_songs,
            })

df_overview_hours = pd.DataFrame(overview_hours)

In [ ]:
# let's first do a quick pre-post - usv
# pre-first day, post-last day
fig, ax = plt.subplots(figsize=(1.5, 1.75))

# get each mouse
df_mouse_pre_post_summary = []
for mouse_1 in df_overview_hours['mouse_1'].unique():
    # get pre and post
    pre = df_overview_hours[(df_overview_hours['condition']=='pre')&(df_overview_hours['mouse_1']==mouse_1)].iloc[0]
    post = df_overview_hours[(df_overview_hours['condition']=='post')&(df_overview_hours['mouse_1']==mouse_1)].iloc[-1]
    df_mouse_pre_post_summary.extend([pre, post])
df_mouse_pre_post_summary = pd.DataFrame(df_mouse_pre_post_summary)

# plot individual mouse
for mouse_1 in df_mouse_pre_post_summary['mouse_1'].unique():
    df_mouse = df_mouse_pre_post_summary[df_mouse_pre_post_summary['mouse_1']==mouse_1]
    ax.plot([0, 1], df_mouse['l_num_usvs'], marker='o', markersize=4, label=mouse_1, color=palette_dict[mouse_1])

# plot mean +- sem
pre_l_mean = df_mouse_pre_post_summary[df_mouse_pre_post_summary['condition']=='pre']['l_num_usvs'].mean()
pre_l_sem = df_mouse_pre_post_summary[df_mouse_pre_post_summary['condition']=='pre']['l_num_usvs'].sem()
post_l_mean = df_mouse_pre_post_summary[df_mouse_pre_post_summary['condition']=='post']['l_num_usvs'].mean()
post_l_sem = df_mouse_pre_post_summary[df_mouse_pre_post_summary['condition']=='post']['l_num_usvs'].sem()

ax.errorbar([0, 1], [pre_l_mean, post_l_mean], yerr=[pre_l_sem, post_l_sem], 
            fmt='-o', markersize=4, capsize=4, color='k')

# annotate plot
ax.legend(loc='upper right', fontsize='x-small')
ax.set_ylim([0, 7000])
ax.set_xticks([0, 1], ['Pre', 'Post'])
ax.set_xlim([-0.5, 1.5])

ax.set_ylabel('USVs / hour')
plt.tight_layout()

# plt.show()

In [ ]:
# let's first do a quick pre-post - song
# pre-first day, post-last day
fig, ax = plt.subplots(figsize=(1.5, 1.75))

# get each mouse
df_mouse_pre_post_summary = []
for mouse_1 in df_overview_hours['mouse_1'].unique():
    # get pre and post
    pre = df_overview_hours[(df_overview_hours['condition']=='pre')&(df_overview_hours['mouse_1']==mouse_1)].iloc[0]
    post = df_overview_hours[(df_overview_hours['condition']=='post')&(df_overview_hours['mouse_1']==mouse_1)].iloc[-1]
    df_mouse_pre_post_summary.extend([pre, post])
df_mouse_pre_post_summary = pd.DataFrame(df_mouse_pre_post_summary)

# plot individual mouse
for mouse_1 in df_mouse_pre_post_summary['mouse_1'].unique():
    df_mouse = df_mouse_pre_post_summary[df_mouse_pre_post_summary['mouse_1']==mouse_1]
    ax.plot([0, 1], df_mouse['l_num_songs'], marker='o', markersize=4, label=mouse_1, color=palette_dict[mouse_1])

# plot mean +- sem
pre_l_mean = df_mouse_pre_post_summary[df_mouse_pre_post_summary['condition']=='pre']['l_num_songs'].mean()
pre_l_sem = df_mouse_pre_post_summary[df_mouse_pre_post_summary['condition']=='pre']['l_num_songs'].sem()
post_l_mean = df_mouse_pre_post_summary[df_mouse_pre_post_summary['condition']=='post']['l_num_songs'].mean()
post_l_sem = df_mouse_pre_post_summary[df_mouse_pre_post_summary['condition']=='post']['l_num_songs'].sem()

print(f'pre song: {pre_l_mean:.2f} +- {pre_l_sem:.2f}')
print(f'post song: {post_l_mean:.2f} +- {post_l_sem:.2f}')

ax.errorbar([0, 1], [pre_l_mean, post_l_mean], yerr=[pre_l_sem, post_l_sem], 
            fmt='-o', markersize=4, capsize=4, color='k')

# annotate plot
ax.legend(loc='upper right', fontsize='x-small')
ax.set_ylim([0, 50])
ax.set_xticks([0, 1], ['Pre', 'Post'])
ax.set_xlim([-0.5, 1.5])

ax.set_ylabel('Songs / hour')
plt.tight_layout()

plt.show()

## Focus on curated hours, analyze how vocalizations are perturbed before going mute

In [ ]:
# analyze temporal patterning
for session in all_sessions:
    print(f'Analyzing session: {session["session_id"]}')
    print(f'Num curated hours: {len(session["curated_hours_l_notes"])}')

    # Process left notes
    processed_curated_left_notes = []
    processed_curated_left_windows = []
    processed_curated_left_filtered_windows = []
    processed_curated_left_segments = []
    processed_curated_left_songs = []

    # store the bouts
    processed_curated_left_vocal_bouts_song = []
    processed_curated_left_vocal_bouts_proto_song = []
    processed_curated_left_vocal_bouts_usv = []
    processed_curated_left_isolated_notes_song = []
    processed_curated_left_isolated_notes_proto_song = []
    processed_curated_left_isolated_notes_usv = []

    for hour_df in session['curated_hours_l_notes']:
        print(f'Num left notes: {len(hour_df)}')
        # compute temporal features - note level
        df_events = compute_event_features(hour_df)
        # compute temporal features - fixed size window
        df_windows = compute_window_features(df_events, min_window_size=min_window_size)
        # join fixed size windows into long segments
        df_filtered_windows, df_segments = merge_stereotypy_windows(df_windows,
                                                                    df_events,
                                                                    max_log_interval_threshold=window_max_log_isi_threshold,
                                                                    rmse_threshold=window_rmse_threshold,
                                                                    max_gap=segment_max_time_gap,
                                                                    max_index_gap=segment_max_index_gap)
        # characterize segments
        df_segments = characterize_segments(df_events, 
                                            df_segments, 
                                            ransac_min_samples=min_window_size,
                                            ransac_normalized_residual_threshold=window_rmse_threshold,)
        # classify segments
        df_segments = classify_segments(df_segments, scaler, pca, gmm, cluster_mapping=cluster_mapping)
        # classify notes by temporal patterning based on the segment it's in
        df_events = classify_notes_by_temporal_patterning(df_segments, df_events)
        # classify notes by loudness
        df_events = classify_notes_by_amplitude(df_events, amp_high_threshold)
        # consolidate vocalizations into final types
        df_events['type_final'] = df_events['type_patterning'].map(final_type_mapping)

        # analyze songs defined by the linear backbone of the song segment
        if df_segments.empty:
            df_songs = pd.DataFrame()
        else:
            df_song_segments = df_segments[df_segments['cluster_label']=='S1']
            # df_song_segments = df_segments[(df_segments['cluster_label']=='S1') | (df_segments['cluster_label']=='S2')]
            df_songs = analyze_song_patterning(df_song_segments, df_events, song_final_note_occupancy=song_final_note_occupancy)

        # bout analysis
        vocal_bouts_song, isolated_notes_song = find_vocal_bouts(df_events[df_events['type_final']=='song'], gap_threshold=0.1)
        vocal_bouts_proto_song, isolated_notes_proto_song = find_vocal_bouts(df_events[df_events['type_final']=='proto-song'], gap_threshold=0.1)
        vocal_bouts_usv, isolated_notes_usv = find_vocal_bouts(df_events[df_events['type_final']=='USV'], gap_threshold=0.1)
        # store the bouts
        processed_curated_left_vocal_bouts_song.append(vocal_bouts_song)
        processed_curated_left_vocal_bouts_proto_song.append(vocal_bouts_proto_song)
        processed_curated_left_vocal_bouts_usv.append(vocal_bouts_usv)
        processed_curated_left_isolated_notes_song.append(isolated_notes_song)
        processed_curated_left_isolated_notes_proto_song.append(isolated_notes_proto_song)
        processed_curated_left_isolated_notes_usv.append(isolated_notes_usv)
        
        processed_curated_left_notes.append(df_events)
        processed_curated_left_windows.append(df_windows)
        processed_curated_left_filtered_windows.append(df_filtered_windows)
        processed_curated_left_segments.append(df_segments)
        processed_curated_left_songs.append(df_songs)

    session['curated_hours_l_notes'] = processed_curated_left_notes
    session['curated_hours_l_windows'] = processed_curated_left_windows
    session['curated_hours_l_filtered_windows'] = processed_curated_left_filtered_windows
    session['curated_hours_l_segments'] = processed_curated_left_segments
    session['curated_hours_l_songs'] = processed_curated_left_songs
    # store the bouts
    session['curated_hours_l_vocal_bouts_song'] = processed_curated_left_vocal_bouts_song
    session['curated_hours_l_vocal_bouts_proto_song'] = processed_curated_left_vocal_bouts_proto_song
    session['curated_hours_l_vocal_bouts_usv'] = processed_curated_left_vocal_bouts_usv
    session['curated_hours_l_isolated_notes_song'] = processed_curated_left_isolated_notes_song
    session['curated_hours_l_isolated_notes_proto_song'] = processed_curated_left_isolated_notes_proto_song
    session['curated_hours_l_isolated_notes_usv'] = processed_curated_left_isolated_notes_usv

In [ ]:
# designate post-early vs post-late
for session in all_sessions:
    if session['condition'] == 'pre':
        continue
    df = session['curated_hours_info']
    total_notes = df['l_num_notes'].sum()
    cut_point = total_notes / 2
    # Compute the cumulative sum of l_num_notes
    cumsum = df['l_num_notes'].cumsum()
    crossing_index = cumsum[cumsum >= cut_point].index[0]
    # Create a new column 'period' based on whether the cumulative sum is <= cut_point
    df['post_period'] = ['early' if idx <= crossing_index else 'late' for idx in df.index]


In [ ]:
# collect all notes
all_curated_hours_notes = []

for session in all_sessions:

    session_id = session['session_id']
    mouse_1 = session['mouse_1']
    mouse_2 = session['mouse_2']
    condition = session['condition']

    curated_hours_info = session['curated_hours_info'].copy()

    print(f'Processing session: {session_id}; mouse_1: {mouse_1}; condition: {condition}')

    for i_hour, curated_hour in curated_hours_info.iterrows():

        df_notes = session['curated_hours_l_notes'][i_hour]

        # add metadata
        df_notes['mouse_id'] = mouse_1
        df_notes['condition'] = condition
        df_notes['session_id'] = session_id
        df_notes['file_idx'] = curated_hour['file_idx']
        if condition == 'pre':
            df_notes['full_condition'] = 'pre'
        else:
            post_period = curated_hour['post_period']
            print(f'file_idx: {curated_hour["file_idx"]}; post_period: {post_period}')
            df_notes['full_condition'] = f'post_{post_period}'

        # save
        all_curated_hours_notes.append(df_notes)

df_all_curated_hours_notes = pd.concat(all_curated_hours_notes, ignore_index=True)

In [ ]:
# collect all local windows
all_curated_hours_windows = []

for session in all_sessions:

    session_id = session['session_id']
    mouse_1 = session['mouse_1']
    mouse_2 = session['mouse_2']
    condition = session['condition']

    curated_hours_info = session['curated_hours_info'].copy()

    print(f'Processing session: {session_id}; mouse_1: {mouse_1}; condition: {condition}')

    for i_hour, curated_hour in curated_hours_info.iterrows():

        df_windows = session['curated_hours_l_windows'][i_hour]

        # add metadata
        df_windows['mouse_id'] = mouse_1
        df_windows['condition'] = condition
        df_windows['session_id'] = session_id
        df_windows['file_idx'] = curated_hour['file_idx']
        if condition == 'pre':
            df_windows['full_condition'] = 'pre'
        else:
            post_period = curated_hour['post_period']
            print(f'file_idx: {curated_hour["file_idx"]}; post_period: {post_period}')
            df_windows['full_condition'] = f'post_{post_period}'

        # save
        all_curated_hours_windows.append(df_windows)

df_all_curated_hours_windows = pd.concat(all_curated_hours_windows, ignore_index=True)

In [ ]:
# collect all segments
all_curated_hours_segments = []

for session in all_sessions:

    session_id = session['session_id']
    mouse_1 = session['mouse_1']
    mouse_2 = session['mouse_2']
    condition = session['condition']

    curated_hours_info = session['curated_hours_info'].copy()

    print(f'Processing session: {session_id}; mouse_1: {mouse_1}; condition: {condition}')

    for i_hour, curated_hour in curated_hours_info.iterrows():
        # print(f'File index: {curated_hour["file_idx"]}; hour offset: {curated_hour["hour_offset"]:.2f}')
        df_segments = session['curated_hours_l_segments'][i_hour]

        # add metadata
        df_segments['mouse_id'] = mouse_1
        df_segments['condition'] = condition
        df_segments['session_id'] = session_id
        df_segments['file_idx'] = curated_hour['file_idx']
        if condition == 'pre':
            df_segments['full_condition'] = 'pre'
        else:
            post_period = curated_hour['post_period']
            print(f'file_idx: {curated_hour["file_idx"]}; post_period: {post_period}')
            df_segments['full_condition'] = f'post_{post_period}'

        # save
        all_curated_hours_segments.append(df_segments)

df_all_curated_hours_segments = pd.concat(all_curated_hours_segments, ignore_index=True)

In [ ]:
# collect all song bouts
all_curated_hours_bouts_song = []

for session in all_sessions:

    session_id = session['session_id']
    mouse_1 = session['mouse_1']
    mouse_2 = session['mouse_2']
    condition = session['condition']

    curated_hours_info = session['curated_hours_info'].copy()

    print(f'Processing session: {session_id}; mouse_1: {mouse_1}; condition: {condition}')

    for i_hour, curated_hour in curated_hours_info.iterrows():
        # print(f'File index: {curated_hour["file_idx"]}; hour offset: {curated_hour["hour_offset"]:.2f}')
        df_bouts = session['curated_hours_l_vocal_bouts_song'][i_hour]

        # add metadata
        df_bouts['mouse_id'] = mouse_1
        df_bouts['condition'] = condition
        df_bouts['session_id'] = session_id
        df_bouts['file_idx'] = curated_hour['file_idx']
        if condition == 'pre':
            df_bouts['full_condition'] = 'pre'
        else:
            post_period = curated_hour['post_period']
            print(f'file_idx: {curated_hour["file_idx"]}; post_period: {post_period}')
            df_bouts['full_condition'] = f'post_{post_period}'

        # save
        all_curated_hours_bouts_song.append(df_bouts)

df_all_curated_hours_bouts_song = pd.concat(all_curated_hours_bouts_song, ignore_index=True)

In [ ]:
# collect all usv bouts
all_curated_hours_bouts_usv = []

for session in all_sessions:

    session_id = session['session_id']
    mouse_1 = session['mouse_1']
    mouse_2 = session['mouse_2']
    condition = session['condition']

    curated_hours_info = session['curated_hours_info'].copy()

    print(f'Processing session: {session_id}; mouse_1: {mouse_1}; condition: {condition}')

    for i_hour, curated_hour in curated_hours_info.iterrows():
        # print(f'File index: {curated_hour["file_idx"]}; hour offset: {curated_hour["hour_offset"]:.2f}')
        df_bouts = session['curated_hours_l_vocal_bouts_usv'][i_hour]

        # add metadata
        df_bouts['mouse_id'] = mouse_1
        df_bouts['condition'] = condition
        df_bouts['session_id'] = session_id
        df_bouts['file_idx'] = curated_hour['file_idx']
        if condition == 'pre':
            df_bouts['full_condition'] = 'pre'
        else:
            post_period = curated_hour['post_period']
            print(f'file_idx: {curated_hour["file_idx"]}; post_period: {post_period}')
            df_bouts['full_condition'] = f'post_{post_period}'

        # save
        all_curated_hours_bouts_usv.append(df_bouts)

df_all_curated_hours_bouts_usv = pd.concat(all_curated_hours_bouts_usv, ignore_index=True)

In [ ]:
# notes
df_all_curated_hours_notes_pre = df_all_curated_hours_notes[df_all_curated_hours_notes['full_condition']=='pre']
df_all_curated_hours_notes_post_early = df_all_curated_hours_notes[df_all_curated_hours_notes['full_condition']=='post_early']
df_all_curated_hours_notes_post_late = df_all_curated_hours_notes[df_all_curated_hours_notes['full_condition']=='post_late']

# windows
df_all_curated_hours_windows_pre = df_all_curated_hours_windows[df_all_curated_hours_windows['full_condition']=='pre']
df_all_curated_hours_windows_post_early = df_all_curated_hours_windows[df_all_curated_hours_windows['full_condition']=='post_early']
df_all_curated_hours_windows_post_late = df_all_curated_hours_windows[df_all_curated_hours_windows['full_condition']=='post_late']

# long segments
df_all_curated_hours_segments_pre = df_all_curated_hours_segments[df_all_curated_hours_segments['full_condition']=='pre']
df_all_curated_hours_segments_post_early = df_all_curated_hours_segments[df_all_curated_hours_segments['full_condition']=='post_early']
df_all_curated_hours_segments_post_late = df_all_curated_hours_segments[df_all_curated_hours_segments['full_condition']=='post_late']

In [ ]:
# local windows - pre

# plot local patterning features - local window nrmse vs gap

# Compute KDE for all notes
data_all = df_all_curated_hours_windows_pre[['window_max_log_inter_event_interval', 'window_rate_index_normalized_rmse']].values.T
x_all, y_all, Z_all = compute_2d_kde(
    data=data_all,
    x_min=-2,
    x_max=1,
    y_min=0,
    y_max=1,
    bw_method='scott',
    gridsize=100,
)

# fig, ax = plt.subplots(figsize=(4,4))
fig, ax = plt.subplots(figsize=(2.25, 2))

# all notes
# kde heatmap
heatmap_all = ax.imshow(
    Z_all.T,
    origin='lower',
    aspect='auto',
    extent=[x_all.min(), x_all.max(), y_all.min(), y_all.max()],
    cmap='Blues',
    vmin=0, vmax=3,
)
cbar_main = plt.colorbar(heatmap_all, ax=ax, label='Density')

# kde contour plot
sns.kdeplot(data=df_all_curated_hours_windows_pre,
            x='window_max_log_inter_event_interval', 
            y='window_rate_index_normalized_rmse',
            thresh=0.1, levels=5, bw_adjust=1,
            clip = [[-2, 1], [0, 1]],
            gridsize=100, fill=False, color='black',
            linewidths=1,
            ax=ax)

# decision boundary for local stereotypy window
ax.add_patch(Rectangle((-2, 0), 
                       window_max_log_isi_threshold-(-2), window_rmse_threshold, linewidth=2, edgecolor='gray', linestyle='--', facecolor='none', zorder=3))

# Set limits and labels for main axes
ax.set_xlim([-2, 1])
ax.set_ylim([0, 1])
# ax.set_title('Social - fixed-size window NRMSE vs max ISI')
ax.set_xlabel('Local Max ISI (log second)')
ax.set_ylabel('Local Normalized RMSE')
plt.tight_layout()
# plt.show()

In [ ]:
# local windows - post early

# plot local patterning features - local window nrmse vs gap

# Compute KDE for all notes
data_all = df_all_curated_hours_windows_post_early[['window_max_log_inter_event_interval', 'window_rate_index_normalized_rmse']].values.T
x_all, y_all, Z_all = compute_2d_kde(
    data=data_all,
    x_min=-2,
    x_max=1,
    y_min=0,
    y_max=1,
    bw_method='scott',
    gridsize=100,
)

# fig, ax = plt.subplots(figsize=(4,4))
fig, ax = plt.subplots(figsize=(2.25, 2))

# all notes
# kde heatmap
heatmap_all = ax.imshow(
    Z_all.T,
    origin='lower',
    aspect='auto',
    extent=[x_all.min(), x_all.max(), y_all.min(), y_all.max()],
    cmap='Oranges',
    vmin=0, vmax=3,
)
cbar_main = plt.colorbar(heatmap_all, ax=ax, label='Density')

# kde contour plot
sns.kdeplot(data=df_all_curated_hours_windows_post_early,
            x='window_max_log_inter_event_interval', 
            y='window_rate_index_normalized_rmse',
            thresh=0.1, levels=5, bw_adjust=1,
            clip = [[-2, 1], [0, 1]],
            gridsize=100, fill=False, color='black',
            linewidths=1,
            ax=ax)

# decision boundary for local stereotypy window
ax.add_patch(Rectangle((-2, 0), 
                       window_max_log_isi_threshold-(-2), window_rmse_threshold, linewidth=2, edgecolor='gray', linestyle='--', facecolor='none', zorder=3))

# Set limits and labels for main axes
ax.set_xlim([-2, 1])
ax.set_ylim([0, 1])
# ax.set_title('Social - fixed-size window NRMSE vs max ISI')
ax.set_xlabel('Local Max ISI (log second)')
ax.set_ylabel('Local Normalized RMSE')
plt.tight_layout()
# plt.show()

In [ ]:
# local windows - post late

# plot local patterning features - local window nrmse vs gap

# Compute KDE for all notes
data_all = df_all_curated_hours_windows_post_late[['window_max_log_inter_event_interval', 'window_rate_index_normalized_rmse']].values.T
x_all, y_all, Z_all = compute_2d_kde(
    data=data_all,
    x_min=-2,
    x_max=1,
    y_min=0,
    y_max=1,
    bw_method='scott',
    gridsize=100,
)

# fig, ax = plt.subplots(figsize=(4,4))
fig, ax = plt.subplots(figsize=(2.25, 2))

# all notes
# kde heatmap
heatmap_all = ax.imshow(
    Z_all.T,
    origin='lower',
    aspect='auto',
    extent=[x_all.min(), x_all.max(), y_all.min(), y_all.max()],
    cmap='Greens',
    vmin=0, vmax=3,
)
cbar_main = plt.colorbar(heatmap_all, ax=ax, label='Density')

# kde contour plot
sns.kdeplot(data=df_all_curated_hours_windows_post_late,
            x='window_max_log_inter_event_interval', 
            y='window_rate_index_normalized_rmse',
            thresh=0.1, levels=5, bw_adjust=1,
            clip = [[-2, 1], [0, 1]],
            gridsize=100, fill=False, color='black',
            linewidths=1,
            ax=ax)

# decision boundary for local stereotypy window
ax.add_patch(Rectangle((-2, 0), 
                       window_max_log_isi_threshold-(-2), window_rmse_threshold, linewidth=2, edgecolor='gray', linestyle='--', facecolor='none', zorder=3))

# Set limits and labels for main axes
ax.set_xlim([-2, 1])
ax.set_ylim([0, 1])
# ax.set_title('Social - fixed-size window NRMSE vs max ISI')
ax.set_xlabel('Local Max ISI (log second)')
ax.set_ylabel('Local Normalized RMSE')
plt.tight_layout()
# plt.show()

In [ ]:
# segments - pre

# Define the feature columns expected by the models
features = ['ransac_num_events', 'ransac_max_rate', 'ransac_min_rate']

# Ensure the dataframe contains the required features
missing_features = [feat for feat in features if feat not in df_all_curated_hours_segments.columns]
if missing_features:
    raise ValueError(f"Missing required features: {missing_features}")

# Select and copy the relevant data
data = df_all_curated_hours_segments_pre[features].copy()

# Apply the scaler and PCA transformations
X_scaled = scaler.transform(data)
X_pca = pca.transform(X_scaled)

# Predict cluster labels using the GMM model
labels = gmm.predict(X_pca)

# Map numeric labels to semantic labels
mapped_labels = np.array([cluster_mapping[label] for label in labels])

# Create a palette mapping for the clusters present in the data
palette_label = {label: color_palette_patterning[label] for label in np.unique(mapped_labels)}

# make 2d-kde
data_all = X_pca.T
x_all, y_all, Z_all = compute_2d_kde(
    data=data_all,
    x_min=-2.75,
    x_max=2.75,
    y_min=-3.5,
    y_max=4.25,
    bw_method='scott',
    gridsize=100,
)

# plot the clusters in pc space
fig, ax = plt.subplots(figsize=(2.25, 2))
# kde heatmap
heatmap_all = ax.imshow(
    Z_all.T,
    origin='lower',
    aspect='auto',
    extent=[x_all.min(), x_all.max(), y_all.min(), y_all.max()],
    cmap='Blues',
    vmin=0, vmax=0.16,
)
cbar_main = plt.colorbar(heatmap_all, ax=ax, label='Density')


sns.kdeplot(x=X_pca[:, 0], y=X_pca[:, 1], 
            color='k', label='pre',
            thresh=0.1, levels=5, bw_adjust=1, linewidths=1, legend=None, ax=ax)
# Overlay ellipses for each Gaussian component for 2σ
for k, (mean, covar) in enumerate(zip(gmm.means_, gmm.covariances_)):
    cluster_label = cluster_mapping[k]  # get the semantic label for component k
    cluster_color = color_palette_patterning[cluster_label]
    draw_ellipse(mean, covar, ax, n_std=1, edgecolor=cluster_color, facecolor='none', lw=1)
    draw_ellipse(mean, covar, ax, n_std=2, edgecolor=cluster_color, facecolor='none', lw=1)
    # draw_ellipse(mean, covar, ax, n_std=3, edgecolor=cluster_color, facecolor='none', lw=0.5)
# # Plot centroids
# for k, mean in enumerate(gmm.means_):
#     ax.scatter(mean[0], mean[1], color='black', marker='X', s=50, edgecolor='none')

# ax.set_title('Backbone of long segments')
# ax.legend(loc='upper left', fontsize='x-small')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_xlim([-2.75, 2.75])
ax.set_ylim([-3.5, 4.25])
plt.tight_layout()
# plt.show()

In [ ]:
# segments - post-early

# Define the feature columns expected by the models
features = ['ransac_num_events', 'ransac_max_rate', 'ransac_min_rate']

# Ensure the dataframe contains the required features
missing_features = [feat for feat in features if feat not in df_all_curated_hours_segments.columns]
if missing_features:
    raise ValueError(f"Missing required features: {missing_features}")

# Select and copy the relevant data
data = df_all_curated_hours_segments_post_early[features].copy()

# Apply the scaler and PCA transformations
X_scaled = scaler.transform(data)
X_pca = pca.transform(X_scaled)

# Predict cluster labels using the GMM model
labels = gmm.predict(X_pca)

# Map numeric labels to semantic labels
mapped_labels = np.array([cluster_mapping[label] for label in labels])

# Create a palette mapping for the clusters present in the data
palette_label = {label: color_palette_patterning[label] for label in np.unique(mapped_labels)}

# make 2d-kde
data_all = X_pca.T
x_all, y_all, Z_all = compute_2d_kde(
    data=data_all,
    x_min=-2.75,
    x_max=2.75,
    y_min=-3.5,
    y_max=4.25,
    bw_method='scott',
    gridsize=100,
)

# plot the clusters in pc space
fig, ax = plt.subplots(figsize=(2.25, 2))
# kde heatmap
heatmap_all = ax.imshow(
    Z_all.T,
    origin='lower',
    aspect='auto',
    extent=[x_all.min(), x_all.max(), y_all.min(), y_all.max()],
    cmap='Oranges',
    vmin=0, vmax=0.16,
)
cbar_main = plt.colorbar(heatmap_all, ax=ax, label='Density')


sns.kdeplot(x=X_pca[:, 0], y=X_pca[:, 1], 
            color='k', label='pre',
            thresh=0.1, levels=5, bw_adjust=1, linewidths=1, legend=None, ax=ax)
# Overlay ellipses for each Gaussian component for 2σ
for k, (mean, covar) in enumerate(zip(gmm.means_, gmm.covariances_)):
    cluster_label = cluster_mapping[k]  # get the semantic label for component k
    cluster_color = color_palette_patterning[cluster_label]
    draw_ellipse(mean, covar, ax, n_std=1, edgecolor=cluster_color, facecolor='none', lw=1)
    draw_ellipse(mean, covar, ax, n_std=2, edgecolor=cluster_color, facecolor='none', lw=1)
    # draw_ellipse(mean, covar, ax, n_std=3, edgecolor=cluster_color, facecolor='none', lw=0.5)
# # Plot centroids
# for k, mean in enumerate(gmm.means_):
#     ax.scatter(mean[0], mean[1], color='black', marker='X', s=50, edgecolor='none')

# ax.set_title('Backbone of long segments')
# ax.legend(loc='upper left', fontsize='x-small')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_xlim([-2.75, 2.75])
ax.set_ylim([-3.5, 4.25])
plt.tight_layout()
# plt.show()

In [ ]:
# segments - post-late

# Define the feature columns expected by the models
features = ['ransac_num_events', 'ransac_max_rate', 'ransac_min_rate']

# Ensure the dataframe contains the required features
missing_features = [feat for feat in features if feat not in df_all_curated_hours_segments.columns]
if missing_features:
    raise ValueError(f"Missing required features: {missing_features}")

# Select and copy the relevant data
data = df_all_curated_hours_segments_post_late[features].copy()

# Apply the scaler and PCA transformations
X_scaled = scaler.transform(data)
X_pca = pca.transform(X_scaled)

# Predict cluster labels using the GMM model
labels = gmm.predict(X_pca)

# Map numeric labels to semantic labels
mapped_labels = np.array([cluster_mapping[label] for label in labels])

# Create a palette mapping for the clusters present in the data
palette_label = {label: color_palette_patterning[label] for label in np.unique(mapped_labels)}

# make 2d-kde
data_all = X_pca.T
x_all, y_all, Z_all = compute_2d_kde(
    data=data_all,
    x_min=-2.75,
    x_max=2.75,
    y_min=-3.5,
    y_max=4.25,
    bw_method='scott',
    gridsize=100,
)

# plot the clusters in pc space
fig, ax = plt.subplots(figsize=(2.25, 2))
# kde heatmap
heatmap_all = ax.imshow(
    Z_all.T,
    origin='lower',
    aspect='auto',
    extent=[x_all.min(), x_all.max(), y_all.min(), y_all.max()],
    cmap='Greens',
    vmin=0, vmax=0.16,
)
cbar_main = plt.colorbar(heatmap_all, ax=ax, label='Density')


sns.kdeplot(x=X_pca[:, 0], y=X_pca[:, 1], 
            color='k', label='pre',
            thresh=0.1, levels=5, bw_adjust=1, linewidths=1, legend=None, ax=ax)
# Overlay ellipses for each Gaussian component for 2σ
for k, (mean, covar) in enumerate(zip(gmm.means_, gmm.covariances_)):
    cluster_label = cluster_mapping[k]  # get the semantic label for component k
    cluster_color = color_palette_patterning[cluster_label]
    draw_ellipse(mean, covar, ax, n_std=1, edgecolor=cluster_color, facecolor='none', lw=1)
    draw_ellipse(mean, covar, ax, n_std=2, edgecolor=cluster_color, facecolor='none', lw=1)
    # draw_ellipse(mean, covar, ax, n_std=3, edgecolor=cluster_color, facecolor='none', lw=0.5)
# # Plot centroids
# for k, mean in enumerate(gmm.means_):
#     ax.scatter(mean[0], mean[1], color='black', marker='X', s=50, edgecolor='none')

# ax.set_title('Backbone of long segments')
# ax.legend(loc='upper left', fontsize='x-small')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_xlim([-2.75, 2.75])
ax.set_ylim([-3.5, 4.25])
plt.tight_layout()
# plt.show()

In [ ]:
# quantify number of song notes, semi-song notes, and USVs, and their proportions in each epoch
vocal_modes_condition_summary = []
for focal_mouse_id in df_all_curated_hours_notes['mouse_id'].unique():
    df_focal_mouse_notes = df_all_curated_hours_notes[df_all_curated_hours_notes['mouse_id']==focal_mouse_id]
    for condition in ['pre', 'post_early', 'post_late']:
        df_focal_mouse_condition_notes = df_focal_mouse_notes[df_focal_mouse_notes['full_condition']==condition]
        num_notes = len(df_focal_mouse_condition_notes)
        num_usvs = len(df_focal_mouse_condition_notes[df_focal_mouse_condition_notes['type_final']=='USV'])
        num_song_notes = len(df_focal_mouse_condition_notes[df_focal_mouse_condition_notes['type_final']=='song'])
        num_semisong_notes = len(df_focal_mouse_condition_notes[df_focal_mouse_condition_notes['type_final']=='proto-song'])

        print(f'focal_mouse_id: {focal_mouse_id}; condition: {condition}; num_notes: {num_notes}; num_song_notes: {num_song_notes}; num_semisong_notes: {num_semisong_notes}; num_usvs: {num_usvs}')
        vocal_modes_condition_summary.append({'mouse_id': focal_mouse_id, 
                                              'condition': condition, 
                                              'num_song_notes': num_song_notes, 
                                              'num_semisong_notes': num_semisong_notes,
                                              'num_usvs': num_usvs, 
                                              'num_notes': num_notes})

df_vocal_modes_condition_summary = pd.DataFrame(vocal_modes_condition_summary)

# compute percentage
df_vocal_modes_condition_summary['perc_song_notes'] = df_vocal_modes_condition_summary['num_song_notes'] / df_vocal_modes_condition_summary['num_notes'] * 100
df_vocal_modes_condition_summary['perc_semisong_notes'] = df_vocal_modes_condition_summary['num_semisong_notes'] / df_vocal_modes_condition_summary['num_notes'] * 100
df_vocal_modes_condition_summary['perc_usvs'] = df_vocal_modes_condition_summary['num_usvs'] / df_vocal_modes_condition_summary['num_notes'] * 100
print('Summary:')
print(f'Sum of USV notes: {df_vocal_modes_condition_summary["num_usvs"].sum()}')
print(f'Sum of Semisong notes: {df_vocal_modes_condition_summary["num_semisong_notes"].sum()}')
print(f'Sum of Song notes: {df_vocal_modes_condition_summary["num_song_notes"].sum()}')
print(f'Sum of all notes: {df_vocal_modes_condition_summary["num_notes"].sum()}')
print(f'Sum of pre notes: {df_vocal_modes_condition_summary[df_vocal_modes_condition_summary["condition"]=="pre"]["num_notes"].sum()}')
print(f'Sum of post-early notes: {df_vocal_modes_condition_summary[df_vocal_modes_condition_summary["condition"]=="post_early"]["num_notes"].sum()}')
print(f'Sum of post-late notes: {df_vocal_modes_condition_summary[df_vocal_modes_condition_summary["condition"]=="post_late"]["num_notes"].sum()}')

fig, ax = plt.subplots(figsize=(1.5, 1.5))

# plot mean and sem percentage of notes as a function of epoch

# usv
pre_perc_usv_mean = df_vocal_modes_condition_summary[df_vocal_modes_condition_summary['condition']=='pre']['perc_usvs'].mean()
pre_perc_usv_sem = df_vocal_modes_condition_summary[df_vocal_modes_condition_summary['condition']=='pre']['perc_usvs'].sem()
post_early_perc_usv_mean = df_vocal_modes_condition_summary[df_vocal_modes_condition_summary['condition']=='post_early']['perc_usvs'].mean()
post_early_perc_usv_sem = df_vocal_modes_condition_summary[df_vocal_modes_condition_summary['condition']=='post_early']['perc_usvs'].sem()
post_late_perc_usv_mean = df_vocal_modes_condition_summary[df_vocal_modes_condition_summary['condition']=='post_late']['perc_usvs'].mean()
post_late_perc_usv_sem = df_vocal_modes_condition_summary[df_vocal_modes_condition_summary['condition']=='post_late']['perc_usvs'].sem()
print(f'Pre USV: {pre_perc_usv_mean:.2f} ± {pre_perc_usv_sem:.2f}')
print(f'Post-early USV: {post_early_perc_usv_mean:.2f} ± {post_early_perc_usv_sem:.2f}')
print(f'Post-late USV: {post_late_perc_usv_mean:.2f} ± {post_late_perc_usv_sem:.2f}')

# semisong notes
pre_perc_semisong_mean = df_vocal_modes_condition_summary[df_vocal_modes_condition_summary['condition']=='pre']['perc_semisong_notes'].mean()
pre_perc_semisong_sem = df_vocal_modes_condition_summary[df_vocal_modes_condition_summary['condition']=='pre']['perc_semisong_notes'].sem()
post_early_perc_semisong_mean = df_vocal_modes_condition_summary[df_vocal_modes_condition_summary['condition']=='post_early']['perc_semisong_notes'].mean()
post_early_perc_semisong_sem = df_vocal_modes_condition_summary[df_vocal_modes_condition_summary['condition']=='post_early']['perc_semisong_notes'].sem()
post_late_perc_semisong_mean = df_vocal_modes_condition_summary[df_vocal_modes_condition_summary['condition']=='post_late']['perc_semisong_notes'].mean()
post_late_perc_semisong_sem = df_vocal_modes_condition_summary[df_vocal_modes_condition_summary['condition']=='post_late']['perc_semisong_notes'].sem()
print(f'Pre semisong: {pre_perc_semisong_mean:.2f} ± {pre_perc_semisong_sem:.2f}')
print(f'Post-early semisong: {post_early_perc_semisong_mean:.2f} ± {post_early_perc_semisong_sem:.2f}')
print(f'Post-late semisong: {post_late_perc_semisong_mean:.2f} ± {post_late_perc_semisong_sem:.2f}')

# song notes
pre_perc_song_mean = df_vocal_modes_condition_summary[df_vocal_modes_condition_summary['condition']=='pre']['perc_song_notes'].mean()
pre_perc_song_sem = df_vocal_modes_condition_summary[df_vocal_modes_condition_summary['condition']=='pre']['perc_song_notes'].sem()
post_early_perc_song_mean = df_vocal_modes_condition_summary[df_vocal_modes_condition_summary['condition']=='post_early']['perc_song_notes'].mean()
post_early_perc_song_sem = df_vocal_modes_condition_summary[df_vocal_modes_condition_summary['condition']=='post_early']['perc_song_notes'].sem()
post_late_perc_song_mean = df_vocal_modes_condition_summary[df_vocal_modes_condition_summary['condition']=='post_late']['perc_song_notes'].mean()
post_late_perc_song_sem = df_vocal_modes_condition_summary[df_vocal_modes_condition_summary['condition']=='post_late']['perc_song_notes'].sem()
print(f'Pre song: {pre_perc_song_mean:.2f} ± {pre_perc_song_sem:.2f}')
print(f'Post-early song: {post_early_perc_song_mean:.2f} ± {post_early_perc_song_sem:.2f}')
print(f'Post-late song: {post_late_perc_song_mean:.2f} ± {post_late_perc_song_sem:.2f}')

# plot
# usv
ax.errorbar([0, 1, 2], [pre_perc_usv_mean, post_early_perc_usv_mean, post_late_perc_usv_mean], 
            yerr=[pre_perc_usv_sem, post_early_perc_usv_sem, post_late_perc_usv_sem], 
            fmt='-o', markersize=2.5, color='tab:purple')
# semisong notes
ax.errorbar([0, 1, 2], [pre_perc_semisong_mean, post_early_perc_semisong_mean, post_late_perc_semisong_mean], 
            yerr=[pre_perc_semisong_sem, post_early_perc_semisong_sem, post_late_perc_semisong_sem], 
            fmt='-o', markersize=2.5, color='tab:blue')
# song notes
ax.errorbar([0, 1, 2], [pre_perc_song_mean, post_early_perc_song_mean, post_late_perc_song_mean],
            yerr=[pre_perc_song_sem, post_early_perc_song_sem, post_late_perc_song_sem],
            fmt='-o', markersize=2.5, color='tab:cyan')

ax.set_ylim([0, 100])
ax.set_yticks([0, 50, 100])
ax.set_ylabel('Vocalizations (%)')
ax.set_xlim([-0.25, 2.25])
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['Pre', 'Post-early', 'Post-late'])

plt.tight_layout()

plt.show()



In [ ]:
# for USVs and song notes, plot loudness of each mode, relative

# USV
df_all_curated_hours_usvs = df_all_curated_hours_notes[df_all_curated_hours_notes['type_final']=='USV'].copy()
print('USVs:')
print(df_all_curated_hours_usvs['full_condition'].value_counts())

# compute relative loudness
pre_usv_amp_mean = df_all_curated_hours_usvs[df_all_curated_hours_usvs['full_condition']=='pre']['max_amp_high'].mean()
print(f'Pre USV mean amp (high): {pre_usv_amp_mean:.2f}')
df_all_curated_hours_usvs['rel_max_amp_high'] = df_all_curated_hours_usvs['max_amp_high'] - pre_usv_amp_mean
pre_usv_rel_amp_mean = df_all_curated_hours_usvs[df_all_curated_hours_usvs['full_condition']=='pre']['rel_max_amp_high'].mean()
pre_usv_rel_amp_sem = df_all_curated_hours_usvs[df_all_curated_hours_usvs['full_condition']=='pre']['rel_max_amp_high'].sem()
post_early_usv_rel_amp_mean = df_all_curated_hours_usvs[df_all_curated_hours_usvs['full_condition']=='post_early']['rel_max_amp_high'].mean()
post_early_usv_rel_amp_sem = df_all_curated_hours_usvs[df_all_curated_hours_usvs['full_condition']=='post_early']['rel_max_amp_high'].sem()
post_late_usv_rel_amp_mean = df_all_curated_hours_usvs[df_all_curated_hours_usvs['full_condition']=='post_late']['rel_max_amp_high'].mean()
post_late_usv_rel_amp_sem = df_all_curated_hours_usvs[df_all_curated_hours_usvs['full_condition']=='post_late']['rel_max_amp_high'].sem()
print(f'USV - Pre: {pre_usv_rel_amp_mean:.2f} ± {pre_usv_rel_amp_sem:.2f}')
print(f'USV - Post Early: {post_early_usv_rel_amp_mean:.2f} ± {post_early_usv_rel_amp_sem:.2f}')
print(f'USV - Post Late: {post_late_usv_rel_amp_mean:.2f} ± {post_late_usv_rel_amp_sem:.2f}')

# song notes
df_all_curated_hours_song_notes = df_all_curated_hours_notes[df_all_curated_hours_notes['type_final']=='song'].copy()
print('Song notes:')
print(df_all_curated_hours_song_notes['full_condition'].value_counts())

pre_song_amp_mean = df_all_curated_hours_song_notes[df_all_curated_hours_song_notes['full_condition']=='pre']['max_amp_song'].mean()
print(f'Pre song mean amp (low): {pre_song_amp_mean:.2f}')
df_all_curated_hours_song_notes['rel_max_amp_song'] = df_all_curated_hours_song_notes['max_amp_song'] - pre_song_amp_mean
pre_song_rel_amp_mean = df_all_curated_hours_song_notes[df_all_curated_hours_song_notes['full_condition']=='pre']['rel_max_amp_song'].mean()
pre_song_rel_amp_sem = df_all_curated_hours_song_notes[df_all_curated_hours_song_notes['full_condition']=='pre']['rel_max_amp_song'].sem()
post_early_song_rel_amp_mean = df_all_curated_hours_song_notes[df_all_curated_hours_song_notes['full_condition']=='post_early']['rel_max_amp_song'].mean()
post_early_song_rel_amp_sem = df_all_curated_hours_song_notes[df_all_curated_hours_song_notes['full_condition']=='post_early']['rel_max_amp_song'].sem()
post_late_song_rel_amp_mean = df_all_curated_hours_song_notes[df_all_curated_hours_song_notes['full_condition']=='post_late']['rel_max_amp_song'].mean()
post_late_song_rel_amp_sem = df_all_curated_hours_song_notes[df_all_curated_hours_song_notes['full_condition']=='post_late']['rel_max_amp_song'].sem()
print(f'Song - Pre: {pre_song_rel_amp_mean:.2f} ± {pre_song_rel_amp_sem:.2f}')
print(f'Song - Post Early: {post_early_song_rel_amp_mean:.2f} ± {post_early_song_rel_amp_sem:.2f}')
print(f'Song - Post Late: {post_late_song_rel_amp_mean:.2f} ± {post_late_song_rel_amp_sem:.2f}')

# plot
fig, axes = plt.subplots(figsize=(2.5, 1.5), nrows=1, ncols=2, sharex=True, sharey=True)

# usv
sns.violinplot(
    x='full_condition',
    y='rel_max_amp_high',
    data=df_all_curated_hours_usvs,
    hue='full_condition',
    palette={'pre':'tab:blue', 'post_early':'tab:orange', 'post_late':'tab:green'},
    ax=axes[0]
)

# song
sns.violinplot(
    x='full_condition',
    y='rel_max_amp_song',
    data=df_all_curated_hours_song_notes,
    hue='full_condition',
    palette={'pre':'tab:blue', 'post_early':'tab:orange', 'post_late':'tab:green'},
    ax=axes[1]
)

axes[0].set_ylabel('Relative amplitude (dB)')
axes[1].set_ylim([-35, 35])
axes[0].set_xlabel('')
axes[1].set_xlabel('')

fig.align_labels(axes)
plt.tight_layout()

plt.show()

## Use song model to analyze perturbed songs

In [ ]:
# collect all songs
all_curated_hours_songs = []

for session in all_sessions:

    session_id = session['session_id']
    mouse_1 = session['mouse_1']
    mouse_2 = session['mouse_2']
    condition = session['condition']

    curated_hours_info = session['curated_hours_info'].copy()

    print(f'Processing session: {session_id}; mouse_1: {mouse_1}; condition: {condition}')

    for i_hour, curated_hour in curated_hours_info.iterrows():

        df_songs = session['curated_hours_l_songs'][i_hour]

        # add metadata
        df_songs['mouse_id'] = mouse_1
        df_songs['condition'] = condition
        df_songs['session_id'] = session_id
        df_songs['file_idx'] = curated_hour['file_idx']
        if condition == 'pre':
            df_songs['full_condition'] = 'pre'
        else:
            post_period = curated_hour['post_period']
            print(f'file_idx: {curated_hour["file_idx"]}; post_period: {post_period}')
            df_songs['full_condition'] = f'post_{post_period}'

        # save
        all_curated_hours_songs.append(df_songs)

df_all_curated_hours_songs = pd.concat(all_curated_hours_songs, ignore_index=True)

In [ ]:
# Compare song duration between male and female
fig, ax = plt.subplots(figsize=(1.75, 1.75))

sns.violinplot(df_all_curated_hours_songs,
              x='full_condition',
              y='duration',
              order=['pre', 'post_early', 'post_late'],
              hue='full_condition', palette={'pre':'tab:blue', 'post_early':'tab:orange', 'post_late':'tab:green'}, hue_order=['pre', 'post_early', 'post_late'],
              ax=ax)

ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['Pre', 'Post-early', 'Post-late'])
ax.set_ylim([0,16])
ax.set_xlabel('')
ax.set_ylabel('Song duration (s)')

plt.tight_layout()

# plt.show()

In [ ]:
print(df_all_curated_hours_songs.groupby('full_condition')[['duration', 'rate_idx_slope', 'rate_idx_rmax', 'rate_idx_rmin']].agg(['count', 'mean', 'sem']).T)

In [ ]:
fig, axes = plt.subplots(figsize=(4, 1.75), ncols=3)

# slope
sns.violinplot(df_all_curated_hours_songs,
              x='full_condition',
              y='rate_idx_slope',
              order=['pre', 'post_early', 'post_late'],
              hue='full_condition', palette={'pre':'tab:blue', 'post_early':'tab:orange', 'post_late':'tab:green'}, hue_order=['pre', 'post_early', 'post_late'],
              legend=False,
              ax=axes[0])
axes[0].set_ylabel('Slope (Hz)')
axes[0].set_ylim([-0.5, 0])
axes[0].set_xlabel('')

# rmax
sns.violinplot(df_all_curated_hours_songs,
              x='full_condition',
              y='rate_idx_rmax',
              order=['pre', 'post_early', 'post_late'],
              hue='full_condition', palette={'pre':'tab:blue', 'post_early':'tab:orange', 'post_late':'tab:green'}, hue_order=['pre', 'post_early', 'post_late'],
              legend=False,
              ax=axes[1])
axes[1].set_ylabel('r_max (Hz)')
axes[1].set_ylim([0, 30])
axes[1].set_xlabel('')

# rmax
sns.violinplot(df_all_curated_hours_songs,
              x='full_condition',
              y='rate_idx_rmin',
              order=['pre', 'post_early', 'post_late'],
              hue='full_condition', palette={'pre':'tab:blue', 'post_early':'tab:orange', 'post_late':'tab:green'}, hue_order=['pre', 'post_early', 'post_late'],
              legend=False,
              ax=axes[2])
axes[2].set_ylabel('r_min (Hz)')
axes[2].set_ylim([0, 30])
axes[2].set_xlabel('')

plt.tight_layout()

# plt.show()

In [ ]:
# Compute the model values as before
mean_rmax = df_all_curated_hours_songs['rate_idx_rmax'].mean()
mean_slope = df_all_curated_hours_songs['rate_idx_slope'].mean()
print(f'Mean r_max: {mean_rmax:.2f}')
print(f'Mean slope: {mean_slope:.4f}')

xs = np.arange(4, 18, 0.1)
ys = 1 / mean_slope * np.log(xs / mean_rmax) + 1 / xs * song_final_note_occupancy

# Create a JointGrid with custom limits
g = sns.JointGrid(data=df_all_curated_hours_songs, 
                  x='rate_idx_rmin', 
                  y='duration', 
                  xlim=(0, 22.5), 
                  ylim=(0, 16), 
                  marginal_ticks=True,
                  height=2, ratio=4)

# Plot the joint scatter plot with hue by full_condition
scatter = sns.scatterplot(data=df_all_curated_hours_songs,
                          x='rate_idx_rmin',
                          y='duration',
                          hue='full_condition',
                          palette={'pre':'tab:blue', 'post_early':'tab:orange', 'post_late':'tab:green'},
                          hue_order=['pre', 'post_early', 'post_late'],
                          s=2,
                          linewidth=0,
                          ax=g.ax_joint)

# Plot the model curve on the joint axes
g.ax_joint.plot(xs, ys, color='k', linewidth=1, linestyle='--', label='Model')

# Add the marginal KDE plots
sns.kdeplot(data=df_all_curated_hours_songs, 
            x='rate_idx_rmin', 
            hue='full_condition',
            palette={'pre':'tab:blue', 'post_early':'tab:orange', 'post_late':'tab:green'},
            hue_order=['pre', 'post_early', 'post_late'],
            fill=True, common_norm=False, legend=None,
            ax=g.ax_marg_x)
sns.kdeplot(data=df_all_curated_hours_songs, 
            y='duration', 
            hue='full_condition',
            palette={'pre':'tab:blue', 'post_early':'tab:orange', 'post_late':'tab:green'},
            hue_order=['pre', 'post_early', 'post_late'],
            fill=True, common_norm=False, legend=None,
            ax=g.ax_marg_y)

# Set axis labels
g.set_axis_labels('r_min (Hz)', 'Song duration (s)')

# Add a legend to the joint plot (adjusting font sizes as needed)
g.ax_joint.legend(title='', fontsize='x-small', title_fontsize='x-small', loc='upper right')

plt.tight_layout()
# plt.show()

In [ ]:
# Compare longest note in each song
fig, ax = plt.subplots(figsize=(1.75, 1.75))

sns.violinplot(df_all_curated_hours_songs,
              x='full_condition',
              y='longest_note_duration',
              order=['pre', 'post_early', 'post_late'],
              hue='full_condition', palette={'pre':'tab:blue', 'post_early':'tab:orange', 'post_late':'tab:green'}, hue_order=['pre', 'post_early', 'post_late'],
              ax=ax)

ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['Pre', 'Post-early', 'Post-late'])
ax.set_ylim([0,0.15])
ax.set_xlabel('')
ax.set_ylabel('Duration of longest note (s)')

plt.tight_layout()
# plt.show()